# Импорт библиотек

In [12]:
import pandas as pd
import requests
import time
from typing import List
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Первоисточник данных:**  
**Московская Биржа (MOEX)**

**Полный путь происхождения данных:**

Участники рынка заключают сделки на Московской Бирже, все сделки фиксируются в торговой системе МОЭКС.
Далее биржа самостоятельно агрегирует сделки в **дневные свечи** (OHLCV) , потом агрегированные данные публикуются через официальный **API ISS MOEX** (`https://iss.moex.com/iss`).Код обращается напрямую к этому API с помощью библиотеки `requests`, полученные JSON-ответы преобразуются в датафрейм


Код, который ниже берёт 80 самых ликвидных акций, проверяет качество исторических данных для каждой. Потом смотрит, чтобы история покрывала не менее 90% от максимальной длины периода, доля пропусков не более 5%. Потом сортирует по ликвидности и выбирает **лучшие 30 акций**.

In [ ]:
# import requests
# import pandas as pd
# import time

# base_url = "https://iss.moex.com/iss"
# start_dt = "2015-01-01"
# end_dt = "2025-12-31"

# session = requests.Session()
# session.headers.update({"User-Agent": "moex-portfolio-project/1.0"})


# def uni_moex_stocks():
#     # какие акции сейчас есть на мосбирже и как они торгуются
#     url = f"{base_url}/engines/stock/markets/shares/boards/TQBR/securities.json"
#     j = session.get(url, timeout=30).json()

#     sec = pd.DataFrame(j["securities"]["data"], columns=j["securities"]["columns"])
#     md = pd.DataFrame(j["marketdata"]["data"], columns=j["marketdata"]["columns"])

#     df = sec.merge(md, on=["SECID", "BOARDID"], how="left", suffixes=("", "_MD"))
#     return df


# def get_candles(ticker, start_date=start_dt, end_date=end_dt, interval=24):
#     # дневные свечи по одной бумаге
#     url = f"{base_url}/engines/stock/markets/shares/securities/{ticker}/candles.json"

#     parts = []
#     start = 0

#     while True:
#         params = {
#             "from": start_date,
#             "till": end_date,
#             "interval": interval,
#             "start": start,
#         }

#         r = session.get(url, params=params, timeout=30)
#         r.raise_for_status()
#         j = r.json()

#         candles = j.get("candles", {})
#         cols = candles.get("columns", [])
#         data = candles.get("data", [])

#         if not data:
#             break

#         chunk = pd.DataFrame(data, columns=cols)
#         chunk["ticker"] = ticker
#         parts.append(chunk)

#         if len(data) < 100:
#             break

#         start += len(data)
#         time.sleep(0.15)

#     if not parts:
#         return pd.DataFrame()

#     df = pd.concat(parts, ignore_index=True)
#     return df


# def build_candidate_list(top_n=80):
#     """
#     Формируем список кандидатов:
#     - активные бумаги
#     - по возможности акции
#     - сортировка по ликвидности
#     """
#     universe = uni_moex_stocks().copy()

#     # Базовый фильтр: активные бумаги
#     if "STATUS" in universe.columns:
#         universe = universe[universe["STATUS"] == "A"].copy()

#     # Оставляем только equity-инструменты, если поле есть
#     if "INSTRID" in universe.columns:
#         universe = universe[universe["INSTRID"] == "EQIN"].copy()

#     # Убираем явный мусор по названию, если встретится
#     if "SHORTNAME" in universe.columns:
#         bad_words = ["ПАЙ", "ETF", "БПИФ", "ИПИФ", "ОБЛ", "OFZ"]
#         mask_bad = universe["SHORTNAME"].fillna("").str.upper().str.contains("|".join(bad_words), regex=True)
#         universe = universe[~mask_bad].copy()

#     # Выбираем метрику ликвидности
#     liquidity_col = None
#     for col in ["VALTODAY_RUR", "VALTODAY", "VALUE"]:
#         if col in universe.columns:
#             liquidity_col = col
#             break

#     if liquidity_col is None:
#         universe["liquidity_proxy"] = 0
#         liquidity_col = "liquidity_proxy"

#     universe[liquidity_col] = pd.to_numeric(universe[liquidity_col], errors="coerce")
#     universe = universe.sort_values(liquidity_col, ascending=False)

#     cols_to_keep = [
#         c for c in [
#             "SECID", "SHORTNAME", "STATUS", "INSTRID", "SECTYPE",
#             "LISTLEVEL", "ISSUECAPITALIZATION", "VALTODAY_RUR", "VALTODAY", "VALUE"
#         ]
#         if c in universe.columns
#     ]

#     return universe[cols_to_keep].head(top_n).reset_index(drop=True)


# def history_quality_report(tickers):
#     """
#     Для каждого тикера считаем качество истории.
#     """
#     reports = []

#     for i, ticker in enumerate(tickers, 1):
#         print(f"[{i}/{len(tickers)}] чекаем {ticker}")
#         try:
#             df = get_candles(ticker)

#             if df.empty:
#                 reports.append({
#                     "ticker": ticker,
#                     "first_date": pd.NaT,
#                     "last_date": pd.NaT,
#                     "obs": 0,
#                     "close_na_share": 1.0,
#                 })
#                 continue

#             if "begin" in df.columns:
#                 df["begin"] = pd.to_datetime(df["begin"], errors="coerce")

#             if "close" in df.columns:
#                 df["close"] = pd.to_numeric(df["close"], errors="coerce")

#             reports.append({
#                 "ticker": ticker,
#                 "first_date": df["begin"].min() if "begin" in df.columns else pd.NaT,
#                 "last_date": df["begin"].max() if "begin" in df.columns else pd.NaT,
#                 "obs": len(df),
#                 "close_na_share": df["close"].isna().mean() if "close" in df.columns else 1.0,
#             })

#         except Exception as e:
#             print(f"  Ошибка {ticker}: {e}")
#             reports.append({
#                 "ticker": ticker,
#                 "first_date": pd.NaT,
#                 "last_date": pd.NaT,
#                 "obs": 0,
#                 "close_na_share": 1.0,
#             })

#         time.sleep(0.2)

#     return pd.DataFrame(reports)


# def select_good_30(exclude_tickers=None):
#     """
#     Полный автоматический отбор 30 хороших акций.
#     exclude_tickers - список тикеров, которые надо исключить.
#     """
#     if exclude_tickers is None:
#         exclude_tickers = []

#     # берем кандидатов
#     candidates = build_candidate_list(top_n=80)
#     tickers = candidates["SECID"].tolist()

#     # проверяем историю
#     quality = history_quality_report(tickers)

#     # Объединяем
#     result = candidates.merge(quality, left_on="SECID", right_on="ticker", how="left")
#     result = result.drop(columns=["ticker"])

#     max_obs = result["obs"].max()
#     min_obs_threshold = int(max_obs * 0.90)

#     # Фильтр качества
#     good = result[
#         (result["obs"] >= min_obs_threshold) &
#         (result["close_na_share"] <= 0.05) &
#         (result["first_date"] <= pd.Timestamp("2015-03-01")) &
#         (result["last_date"] >= pd.Timestamp("2025-12-01"))
#     ].copy()

#     if exclude_tickers:
#         good = good[~good["SECID"].isin(exclude_tickers)].copy()

#     # cортировка по ликвидности
#     sort_cols = [c for c in ["VALTODAY_RUR", "VALTODAY", "VALUE", "ISSUECAPITALIZATION"] if c in good.columns]
#     if sort_cols:
#         good = good.sort_values(sort_cols, ascending=False)
#     else:
#         good = good.sort_values("obs", ascending=False)

#     # Берем первые 30
#     final30 = good.head(30).reset_index(drop=True)

#     return candidates, quality, result, final30


# candidates, quality, result, final30 = select_good_30(exclude_tickers=["IRAO"])

# print("\nИтоговые 30 акций:")
# print(final30[["SECID", "SHORTNAME", "first_date", "last_date", "obs"]])

# # Список тикеров для дальнейшей загрузки
# final_tickers = final30["SECID"].tolist()
# print("\nФинальный список тикеров:")
# print(final_tickers)

# tickers = final30["SECID"].tolist()
# final30.to_csv("moex_final30.csv", index=False, encoding="utf-8-sig")

[1/80] чекаем ROSN
[2/80] чекаем GAZP
[3/80] чекаем SBER
[4/80] чекаем T
[5/80] чекаем NVTK
[6/80] чекаем LENT
[7/80] чекаем YDEX
[8/80] чекаем LKOH
[9/80] чекаем VTBR
[10/80] чекаем PIKK
[11/80] чекаем SMLT
[12/80] чекаем PHOR
[13/80] чекаем PLZL
[14/80] чекаем OZON
[15/80] чекаем X5
[16/80] чекаем GMKN
[17/80] чекаем MAGN
[18/80] чекаем SNGSP
[19/80] чекаем SVCB
[20/80] чекаем RUAL
[21/80] чекаем TATN
[22/80] чекаем AFLT
[23/80] чекаем DOMRF
[24/80] чекаем MGNT
[25/80] чекаем MTSS
[26/80] чекаем SBERP
[27/80] чекаем SNGS
[28/80] чекаем IVAT
[29/80] чекаем AFKS
[30/80] чекаем DATA
[31/80] чекаем TRNFP
[32/80] чекаем VKCO
[33/80] чекаем RNFT
[34/80] чекаем MOEX
[35/80] чекаем CHMF
[36/80] чекаем MTLR
[37/80] чекаем TATNP
[38/80] чекаем EUTR
[39/80] чекаем NLMK
[40/80] чекаем SPBE
[41/80] чекаем UGLD
[42/80] чекаем IRAO
[43/80] чекаем ENPG
[44/80] чекаем SIBN
[45/80] чекаем FLOT
[46/80] чекаем ALRS
[47/80] чекаем HEAD
[48/80] чекаем BSPB
[49/80] чекаем BELU
[50/80] чекаем RENI
[51/80] ч

Далее блок, где для каждого из 30 тикеров последовательно запрашивает дневные свечи (`interval=24`) через API MOEX. Оставляет только нужные колонки (`date`, `ticker`, `close` и др.).
Переводит таблицу из длинного формата (`long`) в широкий (`wide`), где: строки — даты, столбцы — тикеры, значения — цены закрытия

In [ ]:
import requests
import pandas as pd
import time

BASE_URL = "https://iss.moex.com/iss"
START_DATE = "2014-12-31"
END_DATE = "2025-12-31"

TICKERS = final_tickers

session = requests.Session()
session.headers.update({"User-Agent": "moex-portfolio-project/1.0"})


def get_moex_candles(ticker, start_date, end_date, interval=24):
    """
    Скачивает дневные свечи по одной акции с MOEX ISS
    """
    url = f"{BASE_URL}/engines/stock/markets/shares/securities/{ticker}/candles.json"

    all_rows = []
    start = 0

    while True:
        params = {
            "from": start_date,
            "till": end_date,
            "interval": interval,
            "start": start
        }

        response = session.get(url, params=params, timeout=30)
        response.raise_for_status()
        j = response.json()

        candles = j.get("candles", {})
        columns = candles.get("columns", [])
        data = candles.get("data", [])

        if not data:
            break

        chunk = pd.DataFrame(data, columns=columns)
        chunk["ticker"] = ticker
        all_rows.append(chunk)

        if len(data) < 100:
            break

        start += len(data)
        time.sleep(0.2)

    if not all_rows:
        return pd.DataFrame()

    df = pd.concat(all_rows, ignore_index=True)
    return df


def load_all_tickers(tickers, start_date, end_date):
    """
    Скачивает данные по списку тикеров и собирает в одну таблицу
    """
    frames = []

    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] Загружаю {ticker}...")
        try:
            df = get_moex_candles(ticker, start_date, end_date)

            if df.empty:
                print(f"  Нет данных по {ticker}")
                continue

            frames.append(df)

        except Exception as e:
            print(f"  Ошибка для {ticker}: {e}")

        time.sleep(0.3)

    if not frames:
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)
    return result


# загружаем данные
raw = load_all_tickers(TICKERS, START_DATE, END_DATE)

# оставляем только нужные поля
keep_cols = [c for c in ["begin", "ticker", "open", "close", "high", "low", "value", "volume"] if c in raw.columns]
prices = raw[keep_cols].copy()

prices = prices.rename(columns={"begin": "date"})
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
prices = prices[prices["date"].dt.dayofweek < 5].copy()

#  только цены закрытия
close_prices_long = prices[["date", "ticker", "close"]].copy()

print("\nТаблица цен закрытия:")
display(close_prices_long.head())
display(close_prices_long.shape)
close_prices_long.to_csv("moex_close_prices_long_2015_2025.csv", index=False, encoding="utf-8-sig")

close_prices_wide = close_prices_long.pivot(index="date", columns="ticker", values="close").sort_index()

close_prices_wide = close_prices_wide.ffill()
# close_prices_wide = close_prices_wide.dropna(how="all")

print("\nШирокая таблица цен:")
display(close_prices_wide.head())

close_prices_wide.to_csv("moex_close_prices_wide_2015_2025.csv", encoding="utf-8-sig")

[1/30] Загружаю ROSN...
[2/30] Загружаю GAZP...
[3/30] Загружаю SBER...
[4/30] Загружаю NVTK...
[5/30] Загружаю LKOH...
[6/30] Загружаю VTBR...
[7/30] Загружаю PIKK...
[8/30] Загружаю PHOR...
[9/30] Загружаю PLZL...
[10/30] Загружаю GMKN...
[11/30] Загружаю MAGN...
[12/30] Загружаю SNGSP...
[13/30] Загружаю TATN...
[14/30] Загружаю AFLT...
[15/30] Загружаю MGNT...
[16/30] Загружаю MTSS...
[17/30] Загружаю SBERP...
[18/30] Загружаю SNGS...
[19/30] Загружаю AFKS...
[20/30] Загружаю TRNFP...
[21/30] Загружаю MOEX...
[22/30] Загружаю CHMF...
[23/30] Загружаю MTLR...
[24/30] Загружаю TATNP...
[25/30] Загружаю NLMK...
[26/30] Загружаю SIBN...
[27/30] Загружаю ALRS...
[28/30] Загружаю BSPB...
[29/30] Загружаю SELG...
[30/30] Загружаю RTKM...

Таблица цен закрытия:


,date,ticker,close
0,2015-01-05,AFKS,12.25
1,2015-01-06,AFKS,12.37
2,2015-01-08,AFKS,12.60
3,2015-01-09,AFKS,12.61
4,2015-01-12,AFKS,12.13


(82595, 3)


Широкая таблица цен:


ticker,AFKS,AFLT,ALRS,BSPB,CHMF,GAZP,GMKN,LKOH,MAGN,MGNT,...,SBER,SBERP,SELG,SIBN,SNGS,SNGSP,TATN,TATNP,TRNFP,VTBR
date,,,,,,,,,,,,,,,,,,,,,
2015-01-05,12.25,33.21,60.38,24.60,522.00,133.95,85.90,2295.0,11.279,9877.0,...,56.37,38.59,4.595,142.0,24.240,29.600,238.00,134.7,1301.0,337.50
2015-01-06,12.37,33.07,61.28,25.60,556.90,138.92,91.01,2345.0,11.550,10400.0,...,58.28,39.50,4.500,144.6,25.015,30.195,228.75,135.3,1429.9,333.30
2015-01-08,12.60,35.17,60.20,26.50,542.70,146.46,95.50,2572.0,12.350,10627.0,...,65.70,43.75,4.595,148.1,26.265,31.600,245.00,136.0,1300.0,337.05
2015-01-09,12.61,34.00,61.91,26.15,548.55,141.70,97.40,2461.0,12.100,10542.0,...,63.10,42.90,4.555,146.6,25.650,32.010,234.05,134.5,1208.2,326.55
2015-01-12,12.13,34.45,63.00,25.70,558.45,140.22,98.39,2477.0,11.893,10689.0,...,62.90,42.41,4.650,143.1,25.450,31.360,228.25,133.3,1280.0,317.65



Матрица доходностей:


ticker,AFKS,AFLT,ALRS,BSPB,CHMF,GAZP,GMKN,LKOH,MAGN,MGNT,...,SBER,SBERP,SELG,SIBN,SNGS,SNGSP,TATN,TATNP,TRNFP,VTBR
date,,,,,,,,,,,,,,,,,,,,,
2015-01-06,0.009796,-0.004216,0.014906,0.040650,0.066858,0.037103,0.059488,0.021786,0.024027,0.052951,...,0.033883,0.023581,-0.020675,0.018310,0.031972,0.020101,-0.038866,0.004454,0.099078,-0.012444
2015-01-08,0.018593,0.063502,-0.017624,0.035156,-0.025498,0.054276,0.049335,0.096802,0.069264,0.021827,...,0.127316,0.107595,0.021111,0.024205,0.049970,0.046531,0.071038,0.005174,-0.090846,0.011251
2015-01-09,0.000794,-0.033267,0.028405,-0.013208,0.010779,-0.032500,0.019895,-0.043157,-0.020243,-0.007998,...,-0.039574,-0.019429,-0.008705,-0.010128,-0.023415,0.012975,-0.044694,-0.011029,-0.070615,-0.031153
2015-01-12,-0.038065,0.013235,0.017606,-0.017208,0.018048,-0.010445,0.010164,0.006501,-0.017107,0.013944,...,-0.003170,-0.011422,0.020856,-0.023874,-0.007797,-0.020306,-0.024781,-0.008922,0.059427,-0.027255
2015-01-13,0.009893,0.015965,0.053968,-0.005837,0.043961,0.017259,0.018904,0.018167,0.040948,0.045935,...,-0.041176,-0.037491,0.009677,0.004892,0.005894,0.029974,0.028258,0.003751,0.023359,-0.047694


In [16]:
close_prices_wide = pd.read_csv('moex_close_prices_wide_2015_2025.csv', index_col='date', parse_dates=['date'])
close_prices_wide

,AFKS,AFLT,ALRS,BSPB,CHMF,GAZP,GMKN,LKOH,MAGN,MGNT,...,SBER,SBERP,SELG,SIBN,SNGS,SNGSP,TATN,TATNP,TRNFP,VTBR
date,,,,,,,,,,,,,,,,,,,,,
2015-01-05,12.250,33.21,60.38,24.60,522.00,133.95,85.90,2295.0,11.279,9877.0,...,56.37,38.59,4.595,142.00,24.240,29.600,238.00,134.7,1301.0,337.50
2015-01-06,12.370,33.07,61.28,25.60,556.90,138.92,91.01,2345.0,11.550,10400.0,...,58.28,39.50,4.500,144.60,25.015,30.195,228.75,135.3,1429.9,333.30
2015-01-08,12.600,35.17,60.20,26.50,542.70,146.46,95.50,2572.0,12.350,10627.0,...,65.70,43.75,4.595,148.10,26.265,31.600,245.00,136.0,1300.0,337.05
2015-01-09,12.610,34.00,61.91,26.15,548.55,141.70,97.40,2461.0,12.100,10542.0,...,63.10,42.90,4.555,146.60,25.650,32.010,234.05,134.5,1208.2,326.55
2015-01-12,12.130,34.45,63.00,25.70,558.45,140.22,98.39,2477.0,11.893,10689.0,...,62.90,42.41,4.650,143.10,25.450,31.360,228.25,133.3,1280.0,317.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,13.197,57.87,42.36,310.73,965.80,125.04,145.72,5677.5,28.020,3036.5,...,302.03,299.84,44.700,487.55,22.135,41.220,572.90,531.2,1353.8,71.60
2025-12-25,13.087,56.90,42.34,311.04,964.20,124.20,147.16,5673.5,27.895,3001.0,...,300.26,297.62,43.850,482.10,21.920,40.600,568.80,529.4,1349.6,71.40
2025-12-26,13.337,58.85,43.00,313.42,980.00,127.37,152.00,5790.0,28.550,3042.5,...,302.51,300.09,44.400,489.00,22.150,40.780,584.30,541.0,1390.8,72.92


# Пункт 2

### 2.1 Матрица доходностей

Доходность акции в момент времени $t$ определяется как

$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$

где $P_t$ — цена закрытия в день $t$

In [18]:
close_prices_wide = close_prices_wide.sort_index() # для начала отсортируем даты
returns = close_prices_wide.pct_change() # считаем простые дневные доходности
returns = returns.iloc[1:].copy()

display("Размер матрицы доходностей:", returns.shape)
display(returns.head())


'Размер матрицы доходностей:'

(2753, 30)

,AFKS,AFLT,ALRS,BSPB,CHMF,GAZP,GMKN,LKOH,MAGN,MGNT,...,SBER,SBERP,SELG,SIBN,SNGS,SNGSP,TATN,TATNP,TRNFP,VTBR
date,,,,,,,,,,,,,,,,,,,,,
2015-01-06,0.009796,-0.004216,0.014906,0.040650,0.066858,0.037103,0.059488,0.021786,0.024027,0.052951,...,0.033883,0.023581,-0.020675,0.018310,0.031972,0.020101,-0.038866,0.004454,0.099078,-0.012444
2015-01-08,0.018593,0.063502,-0.017624,0.035156,-0.025498,0.054276,0.049335,0.096802,0.069264,0.021827,...,0.127316,0.107595,0.021111,0.024205,0.049970,0.046531,0.071038,0.005174,-0.090846,0.011251
2015-01-09,0.000794,-0.033267,0.028405,-0.013208,0.010779,-0.032500,0.019895,-0.043157,-0.020243,-0.007998,...,-0.039574,-0.019429,-0.008705,-0.010128,-0.023415,0.012975,-0.044694,-0.011029,-0.070615,-0.031153
2015-01-12,-0.038065,0.013235,0.017606,-0.017208,0.018048,-0.010445,0.010164,0.006501,-0.017107,0.013944,...,-0.003170,-0.011422,0.020856,-0.023874,-0.007797,-0.020306,-0.024781,-0.008922,0.059427,-0.027255
2015-01-13,0.009893,0.015965,0.053968,-0.005837,0.043961,0.017259,0.018904,0.018167,0.040948,0.045935,...,-0.041176,-0.037491,0.009677,0.004892,0.005894,0.029974,0.028258,0.003751,0.023359,-0.047694


In [19]:
(returns.groupby(returns.index.to_period("W")).size()).mean()

np.float64(4.8213660245183885)

## Расчет доходностей и ковариаций

В данном блоке рассчитываются:

- вектор средних доходностей \($mu_t$);
- ковариационная матрица \($Sigma_t$);

для набора акций на разные даты.

### Подход

Расчеты выполняются на выбранных датах (год, квартал, месяц и т.д.) двумя способами:

1. **Скользящее окно**  
   Используются только последние наблюдения фиксированной длины (например, 1 год ≈ 252 дня).

2. **Расширяющееся окно**  
   Используются все доступные данные от начала выборки до текущей даты.

---

### Оценки

Для каждого окна считаются:

- средние доходности (вектор \($mu_t$));
- ковариации доходностей (матрица \($Sigma_t$)).

---

### Взвешивание

Рассматриваются два варианта:

- **без весов** — все наблюдения равнозначны;
- **с экспоненциальным забыванием** — более свежие данные имеют больший вес (параметр \($lambda$)).

---

### Структура кода

- `get_rebalance_dates` — выбирает даты расчета;
- `mean_cov_unweighted` — обычные оценки;
- `mean_cov_ew` — оценки с экспоненциальным забыванием;
- `rolling_mean_cov` — расчеты на скользящем окне;
- `expanding_mean_cov` — расчеты на расширяющемся окне.

In [21]:
def get_rebalance_dates(index, step="Y"):
    """
    Выбирает даты, на которых считаем mu и Sigma.
    'Y'  - конец года
    'Q'  - конец квартала
    'M'  - конец месяца
    'W'  - конец недели
    'D'  - каждый день
    """
    idx = pd.DatetimeIndex(index)

    if step == "D":
        return idx

    s = pd.Series(index=idx, data=1)
    dates = s.resample(step).last().dropna().index
    return dates

# Обычные mean/cov на заданном окне

def mean_cov_unweighted(window_returns):
    """
    window_returns: DataFrame (T x N)
    Возвращает:
        mu    : Series (N,)
        Sigma : DataFrame (N x N)
    """
    mu = window_returns.mean()
    Sigma = window_returns.cov()
    return mu, Sigma

# Mean/cov с экспоненциальным забыванием

def exp_weights(n, lam=0.94):
    """
    n - число наблюдений
    lam - коэффициент забывания
    Вес самого свежего наблюдения максимален
    """
    # k=0 для самого старого, поэтому развернем позже
    w = np.array([lam**k for k in range(n-1, -1, -1)], dtype=float)
    w /= w.sum()
    return w

def mean_cov_ew(window_returns, lam=0.94):
    """
    Экспоненциально-взвешенные mean и covariance
    """
    X = window_returns.values  # shape (T, N)
    n, m = X.shape

    w = exp_weights(n, lam=lam)  # shape (T,)

    mu = np.sum(X * w[:, None], axis=0)  # shape (N,)
    X_centered = X - mu

    Sigma = (X_centered * w[:, None]).T @ X_centered

    mu = pd.Series(mu, index=window_returns.columns, name="mu")
    Sigma = pd.DataFrame(Sigma, index=window_returns.columns, columns=window_returns.columns)

    return mu, Sigma


# Скользящее окно длиной 1 год

def rolling_mean_cov(
    returns,
    window_days=252,
    step="Y",
    weighted=False,
    lam=0.94
):

    rebalance_dates = get_rebalance_dates(returns.index, step=step)

    mus = {}
    covs = {}

    for dt in rebalance_dates:
        # Берем все данные до dt включительно
        sub = returns.loc[:dt]
        if len(sub) < window_days:
            continue

        window = sub.iloc[-window_days:]

        if weighted:
            mu, Sigma = mean_cov_ew(window, lam=lam)
        else:
            mu, Sigma = mean_cov_unweighted(window)

        mus[dt] = mu
        covs[dt] = Sigma

    return mus, covs


# Расшир окно

def expanding_mean_cov(
    returns,
    min_days = 252,
    step = "Y",
    weighted=False,
    lam = 0.94
):
    """
    expanding window: от начала выборки до даты dt
    min_days: минимальное число наблюдений перед первым расчетом
    """
    rebalance_dates = get_rebalance_dates(returns.index, step=step)

    mus = {}
    covs = {}

    for dt in rebalance_dates:
        window = returns.loc[:dt]

        if len(window) < min_days:
            continue

        if weighted:
            mu, Sigma = mean_cov_ew(window, lam=lam)
        else:
            mu, Sigma = mean_cov_unweighted(window)

        mus[dt] = mu
        covs[dt] = Sigma

    return mus, covs

Скользящее окно 1 год, шаг 1 год

In [23]:
mus_roll_y, covs_roll_y = rolling_mean_cov(
    returns,
    window_days=252,   # примерно 1 торговый год, брал среднее по всем торговым годам, которые попали в выборку
    step="Y",
    weighted=False
)
display(mus_roll_y)

/tmp/ipykernel_14622/2111241458.py:18: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = s.resample(step).last().dropna().index


{Timestamp('2016-12-31 00:00:00'): AFKS     0.001205
 AFLT     0.004196
 ALRS     0.002425
 BSPB     0.001786
 CHMF     0.001872
 GAZP     0.000565
 GMKN     0.000495
 LKOH     0.001645
 MAGN     0.002682
 MGNT     0.000195
 MOEX     0.001336
 MTLR     0.004659
 MTSS     0.000919
 MVID     0.001812
 NLMK     0.002613
 NVTK     0.001224
 PHOR    -0.000231
 PIKK     0.001193
 PLZL     0.001779
 ROSN     0.001960
 SBER     0.002272
 SBERP    0.002227
 SELG     0.003509
 SIBN     0.001369
 SNGS    -0.000342
 SNGSP   -0.001110
 TATN     0.001318
 TATNP    0.000863
 TRNFP    0.000500
 VTBR    -0.000100
 dtype: float64,
 Timestamp('2017-12-31 00:00:00'): AFKS    -0.001632
 AFLT    -0.000196
 ALRS    -0.000882
 BSPB    -0.000682
 CHMF    -0.000119
 GAZP    -0.000610
 GMKN     0.000388
 LKOH    -0.000049
 MAGN     0.001087
 MGNT    -0.002000
 MOEX    -0.000412
 MTLR    -0.000466
 MTSS     0.000407
 MVID     0.000354
 NLMK     0.001117
 NVTK    -0.000529
 PHOR    -0.000067
 PIKK     0.000574
 PL

Скользящее окно 1 квартал / 1 месяц / 1 неделя / 1 день

In [24]:
mus_roll_q, covs_roll_q = rolling_mean_cov(returns, window_days=63, step="Q", weighted=False)
mus_roll_m, covs_roll_m = rolling_mean_cov(returns, window_days=21, step="M", weighted=False)
mus_roll_w, covs_roll_w = rolling_mean_cov(returns, window_days=5,  step="W", weighted=False)
mus_roll_d, covs_roll_d = rolling_mean_cov(returns, window_days=1,  step="D", weighted=False)

/tmp/ipykernel_14622/2111241458.py:18: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  dates = s.resample(step).last().dropna().index
/tmp/ipykernel_14622/2111241458.py:18: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = s.resample(step).last().dropna().index
/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py:11211: RuntimeWarning: Degrees of freedom <= 0 for slice
  base_cov = np.cov(mat.T, ddof=ddof)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


Расширяющееся окно, шаг 1 год

In [25]:
mus_exp_y, covs_exp_y = expanding_mean_cov(
    returns,
    min_days=252,
    step="Y",
    weighted=False
)

/tmp/ipykernel_14622/2111241458.py:18: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = s.resample(step).last().dropna().index


То же, но с экспоненциальным забыванием

In [26]:
mus_roll_y_ew, covs_roll_y_ew = rolling_mean_cov(
    returns,
    window_days=252,
    step="Y",
    weighted=True,
    lam=0.94
)

mus_exp_y_ew, covs_exp_y_ew = expanding_mean_cov(
    returns,
    min_days=252,
    step="Y",
    weighted=True,
    lam=0.94
)

/tmp/ipykernel_14622/2111241458.py:18: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  dates = s.resample(step).last().dropna().index


In [27]:
# векторы доходностей
mu_table = pd.DataFrame(mus_roll_y).T
mu_table.index.name = "Date"
print(mu_table.head())

# пример ковариационной матрицы
some_date = list(covs_roll_y.keys())[0]
print("Дата:", some_date)
print(covs_roll_y[some_date])

                AFKS      AFLT      ALRS      BSPB      CHMF      GAZP  \
Date                                                                     
2016-12-31  0.001205  0.004196  0.002425  0.001786  0.001872  0.000565   
2017-12-31 -0.001632 -0.000196 -0.000882 -0.000682 -0.000119 -0.000610   
2018-12-31 -0.001510 -0.001075  0.001180 -0.000836  0.000324  0.000750   
2019-12-31  0.002677  0.000165 -0.000508  0.001114  0.000070  0.002219   
2020-12-31  0.002913 -0.001130  0.000927  0.000207  0.001537 -0.000524   

                GMKN      LKOH          MAGN      MGNT  ...      SBER  \
Date                                                    ...             
2016-12-31  0.000495  0.001645  2.681743e-03  0.000195  ...  0.002272   
2017-12-31  0.000388 -0.000049  1.086753e-03 -0.002000  ...  0.001153   
2018-12-31  0.000980  0.001723  1.806925e-04 -0.002125  ... -0.000462   
2019-12-31  0.001595  0.000946  3.426455e-07  0.000021  ...  0.001322   
2020-12-31  0.001103 -0.000242  1.393765e-0

In [28]:
# сохраним в эксель
with pd.ExcelWriter("results.xlsx") as writer:
    pd.DataFrame(mus_roll_y).T.to_excel(writer, sheet_name="mu_roll_y")
    pd.DataFrame(mus_exp_y).T.to_excel(writer, sheet_name="mu_expand_y")


    for dt, cov in covs_roll_y.items():
        sheet = f"cov_roll_{dt.strftime('%Y%m%d')}"[:31]
        cov.to_excel(writer, sheet_name=sheet)